In [2]:
# Import Libraries
import pandas as pd

In [3]:
# load the CSV file
coingecko_global = pd.read_csv('files\\coingecko_global.csv')
coingecko_btc = pd.read_csv('files\\coingecko_btc.csv')


# Convert global index from milliseconds to datetime
coingecko_global['snapped_at'] = pd.to_datetime(coingecko_global['snapped_at'], unit='ms')
coingecko_global = coingecko_global.rename(columns={'snapped_at': 'timestamp'})
coingecko_global = coingecko_global.set_index('timestamp')


# Convert BTC index to datetime
coingecko_btc['snapped_at'] = pd.to_datetime(coingecko_btc['snapped_at'], utc=False)
coingecko_btc = coingecko_btc.rename(columns={'snapped_at': 'timestamp'})
coingecko_btc = coingecko_btc.set_index('timestamp')

# Ensure both indices are timezone-naive
coingecko_global.index = coingecko_global.index.tz_localize(None)
coingecko_btc.index = coingecko_btc.index.tz_localize(None)

# Resampling example - weekly average market cap
w_coingecko_global = coingecko_global['market_cap'].resample('W').mean()
w_coingecko_btc = coingecko_btc['market_cap'].resample('W').mean()


pd.to_pickle(w_coingecko_global, 'files\\coingecko_global.pkl')
pd.to_pickle(w_coingecko_btc, 'files\\coingecko_btc.pkl')

In [5]:
# Create DataFrames from the weekly resampled series
global_df = w_coingecko_global.to_frame(name="global_mcap")
btc_df = w_coingecko_btc.to_frame(name="btc_mcap")

# Merge on the timestamp index (inner join)
mcap_df = global_df.merge(btc_df, left_index=True, right_index=True, how="inner")

# Compute the difference (global minus BTC)
mcap_df["total2_mcap"] = mcap_df["global_mcap"] - mcap_df["btc_mcap"]

# Filter rows with index after 2017
mcap_df = mcap_df[mcap_df.index > "2017-01-01"]

# Filter rows with index after 2017 and round to 2 decimals
mcap_df = mcap_df[mcap_df.index > "2017-01-01"].astype(int)

pd.to_pickle(mcap_df, 'files\\mcap_df.pkl')